In [15]:
import gc
import glob
import os
import re
import sys

import numpy as np
import optuna
import pandas as pd
import torch
from tqdm import tqdm

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [16]:
device.type

'cuda'

In [17]:
%load_ext autoreload
%autoreload 2

from drGAT import No_atten_drGAT

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
def load_and_combine_chunks(pattern, axis=0):
    chunk_files = sorted(
        glob.glob(pattern), key=lambda x: int(x.split("_")[-1].split(".")[0])
    )

    chunks = [np.load(f) for f in chunk_files]
    return np.concatenate(chunks, axis=axis)


edge_index = load_and_combine_chunks("../nci_data/edge_idxs/*.npy", axis=1)
edge_attr = load_and_combine_chunks("../nci_data/edge_attrs/*.npy", axis=0)
edge_index = torch.tensor(edge_index).int()
edge_index = edge_index.type(torch.int64)
edge_attr = torch.tensor(edge_attr).float()

In [19]:
idxs = np.load("../nci_data/idxs.npy", allow_pickle=True)
converter = {idxs[1, i]: int(idxs[0, i]) for i in range(idxs.shape[1])}

In [20]:
def get_idx(X):
    X["Drug"] = [converter[(i)] for i in X["Drug"]]
    X["Cell"] = [converter[(i)] for i in X["Cell"]]
    return X

In [21]:
train_data = pd.read_csv("../nci_data/train.csv")
val_data = pd.read_csv("../nci_data/val.csv")
test_data = pd.read_csv("../nci_data/test.csv")
train_data = get_idx(train_data)
val_data = get_idx(val_data)
test_data = get_idx(test_data)

In [22]:
train_drug = train_data["Drug"].values
train_cell = train_data["Cell"].values
val_drug = val_data["Drug"].values
val_cell = val_data["Cell"].values

In [23]:
train_labels = np.load("../nci_data/train_labels.npy")
val_labels = np.load("../nci_data/val_labels.npy")

train_labels = torch.tensor(train_labels).float()
val_labels = torch.tensor(val_labels).float()

## Get feature matrix

In [24]:
cell = pd.read_csv("../nci_data/cell_sim.csv", index_col=0)


# How to read
def natural_sort_key(s):
    return [int(c) if c.isdigit() else c for c in re.split(r"(\d+)", s)]


file_paths = glob.glob("../nci_data/drug_sim/drug_sim_part_*.parquet")
sorted_file_paths = sorted(file_paths, key=natural_sort_key)

drug = pd.concat([pd.read_parquet(file) for file in tqdm(sorted_file_paths)])


file_paths = glob.glob("../nci_data/gene_sim/gene_sim_part_*.parquet")
sorted_file_paths = sorted(file_paths, key=natural_sort_key)

gene = pd.concat([pd.read_parquet(file) for file in tqdm(sorted_file_paths)])

100%|██████████| 25/25 [00:02<00:00, 11.49it/s]


In [25]:
gene

,A2M,ABCA3,ABCB1,ABCB5,ABCC1,ABCC3,ABCC5,ABCD1,ABHD17C,ABHD4,...,ZNF358,ZNF462,ZNF542P,ZNF655,ZNF667-AS1,ZNF703,ZNRD1,ZP3,ZSCAN18,ZYX
A2M,1.000000,0.060190,0.090301,0.391793,0.099959,0.062216,0.166070,0.340668,0.138102,0.156964,...,0.185882,0.210333,0.128977,0.165382,0.119324,0.222604,0.069420,0.076629,0.097455,0.140388
ABCA3,0.060190,1.000000,0.275980,0.048084,0.215076,0.174623,0.159356,0.058527,0.087342,0.135836,...,0.194404,0.154588,0.187560,0.078862,0.244293,0.101630,0.123893,0.361435,0.203343,0.167400
ABCB1,0.090301,0.275980,1.000000,0.087243,0.240955,0.132595,0.166354,0.080465,0.104796,0.189396,...,0.176134,0.141762,0.229688,0.131027,0.180989,0.128992,0.187949,0.200042,0.174078,0.223905
ABCB5,0.391793,0.048084,0.087243,1.000000,0.063024,0.041796,0.164287,0.373311,0.186304,0.119026,...,0.162311,0.170567,0.114037,0.193312,0.068295,0.224850,0.086748,0.064751,0.059511,0.115838
ABCC1,0.099959,0.215076,0.240955,0.063024,1.000000,0.349966,0.127889,0.114741,0.122463,0.340527,...,0.202791,0.227102,0.182738,0.139628,0.134596,0.157619,0.154868,0.230891,0.160625,0.333414
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF703,0.222604,0.101630,0.128992,0.224850,0.157619,0.175506,0.071109,0.230201,0.185125,0.284584,...,0.141732,0.190102,0.101912,0.068699,0.103096,1.000000,0.058706,0.178124,0.050863,0.194664
ZNRD1,0.069420,0.123893,0.187949,0.086748,0.154868,0.054900,0.418163,0.054293,0.126994,0.045645,...,0.055761,0.058149,0.244632,0.189314,0.128546,0.058706,1.000000,0.126709,0.250182,0.083467
ZP3,0.076629,0.361435,0.200042,0.064751,0.230891,0.211739,0.172942,0.119484,0.133095,0.218610,...,0.155753,0.153724,0.125526,0.079986,0.181292,0.178124,0.126709,1.000000,0.159002,0.197240
ZSCAN18,0.097455,0.203343,0.174078,0.059511,0.160625,0.085757,0.245629,0.075171,0.062191,0.103212,...,0.188774,0.133152,0.415823,0.193341,0.380405,0.050863,0.250182,0.159002,1.000000,0.135090


In [26]:
drug = torch.tensor(drug.values).float()
cell = torch.tensor(cell.values).float()
gene = torch.tensor(gene.values).float()

# Create the dataset

In [27]:
data = [
    drug,
    cell,
    gene,
    edge_index,
    edge_attr,
    train_drug,
    train_cell,
    val_drug,
    val_cell,
    train_labels,
    val_labels,
]

In [31]:
def objective(trial):
    params = {
        "n_drug": drug.shape[0],
        "n_cell": cell.shape[0],
        "n_gene": gene.shape[0],
        "dropout1": trial.suggest_categorical("dropout1", [0.1, 0.2, 0.3, 0.4, 0.5]),
        "dropout2": trial.suggest_categorical("dropout2", [0.1, 0.2, 0.3, 0.4, 0.5]),
        "hidden1": trial.suggest_categorical(
            "hidden1",
            [256, 512, 1028],
        ),
        "hidden2": trial.suggest_categorical(
            "hidden2",
            [
                128,
                256,
                512,
            ],
        ),
        "hidden3": trial.suggest_categorical(
            "hidden3",
            [
                64,
                128,
                256,
            ],
        ),
        "epochs": trial.suggest_int("epochs", 10, 200, step=50),
        "activation": trial.suggest_categorical(
            "activation", ["relu", "gelu", "swish"]
        ),
        "optimizer": trial.suggest_categorical("optimizer", ["Adam", "AdamW", "SGD"]),
        "lr": trial.suggest_float("lr", 1e-5, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "scheduler": trial.suggest_categorical(
            "scheduler", [None, "Cosine", "Step", "Plateau"]
        ),
        "gnn_layer": trial.suggest_categorical(
            "gnn_layer",
            ["GCN", "MPNN"],
        ),
    }

    if params["scheduler"] == "Cosine":
        params["T_max"] = trial.suggest_int("T_max", 20, 50)
    elif params["scheduler"] == "Step":
        params["scheduler_gamma"] = trial.suggest_float("gamma_step", 0.1, 0.95)
        params["step_size"] = trial.suggest_int("step_size", 10, 30)
    elif params["scheduler"] == "Plateau":
        params["patience"] = trial.suggest_int("patience_plateau", 3, 10)
        params["threshold"] = trial.suggest_float(
            "thresh_plateau", 1e-4, 1e-2, log=True
        )

    if params["hidden1"] < params["hidden2"] or params["hidden2"] < params["hidden3"]:
        raise optuna.TrialPruned("Invalid layer size configuration")

    if params["optimizer"] in ["Adam", "AdamW"]:
        params["amsgrad"] = trial.suggest_categorical("amsgrad", [True, False])

    if params["optimizer"] == "SGD":
        params["momentum"] = trial.suggest_float("momentum", 0.8, 0.95)
        params["nesterov"] = trial.suggest_categorical("nesterov", [True, False])

    if (params["hidden1"] > 512) and (params["hidden2"] > 256):
        raise optuna.TrialPruned("Memory intensive configuration")

    try:
        _, best_metrics, early_stopping_epoch = No_atten_drGAT.train(
            data, params=params, device=device, verbose=False
        )

        early_stop_threshold = trial.suggest_float("early_stop_threshold", 0.3, 0.7)
        if (
            early_stopping_epoch is not None
            and early_stopping_epoch < params["epochs"] * early_stop_threshold
        ):
            raise optuna.TrialPruned("Early stopping occurred too early")

        trial.set_user_attr("early_stopping_epoch", early_stopping_epoch)
        return best_metrics

    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            print(f"Pruned trial {trial.number}: CUDA OOM")

            # メモリ解放処理
            with torch.cuda.device("cuda"):
                torch.cuda.empty_cache()
            gc.collect()

            # Pruning通知
            raise optuna.TrialPruned(f"OOM at trial {trial.number}")

        else:
            raise e

    except ValueError as e:
        if "Input contains NaN" in str(e):
            print(f"Pruned trial {trial.number}: Input contains NaN")

            # Pruning通知
            raise optuna.TrialPruned(f"NaN input at trial {trial.number}")
        else:
            raise e

In [32]:
name = "NCI"
study = optuna.create_study(
    directions=["maximize"] * 4,
    sampler=optuna.samplers.TPESampler(),
    pruner=optuna.pruners.HyperbandPruner(),
    storage="sqlite:///{}_{}.sqlite3".format(name, "GCN_MPNN"),
    study_name=name,
    load_if_exists=True,
)
study.optimize(objective, n_trials=100)

[I 2025-03-22 17:11:14,243] Using an existing study with name 'NCI' instead of creating a new one.
/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/optuna/distributions.py:700: UserWarning: The distribution is specified by [10, 200] and step=50, but the range is not divisible by `step`. It will be replaced by [10, 160].
  warnings.warn(
[I 2025-03-22 17:11:29,719] Trial 290 pruned. Memory intensive configuration


Using:  cuda


  0%|          | 0/10 [00:00<?, ?it/s]

After MPNN Layer 1: False
After Dropout Layer 1: False
After MPNN Layer 2: False
After Dropout Layer 2: False


/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:216: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
 10%|█         | 1/10 [00:00<00:06,  1.49it/s]

...

After MPNN Layer 1: False
After Dropout Layer 1: False
After MPNN Layer 2: False
After Dropout Layer 2: False
After MPNN Layer 1: False
After Dropout Layer 1: False
After MPNN Layer 2: False
After Dropout Layer 2: False


100%|██████████| 10/10 [00:06<00:00,  1.55it/s]

After MPNN Layer 1: False
After Dropout Layer 1: False
After MPNN Layer 2: False
After Dropout Layer 2: False
Best model found at epoch 10



[I 2025-03-22 17:13:57,726] Trial 321 finished with values: [0.5, 0.6559025175420753, 0.677237646276622, 0.0] and parameters: {'dropout1': 0.4, 'dropout2': 0.1, 'hidden1': 1028, 'hidden2': 128, 'hidden3': 64, 'epochs': 10, 'activation': 'swish', 'optimizer': 'Adam', 'lr': 0.0009194521342437521, 'weight_decay': 0.0023669315411051716, 'scheduler': 'Cosine', 'gnn_layer': 'MPNN', 'T_max': 47, 'amsgrad': False, 'early_stop_threshold': 0.4011715215806287}. 
[W 2025-03-22 17:13:58,316] Trial 323 failed with parameters: {} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/optuna/study/_optimize.py", line 196, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_3537163/515194571.py", line 6, in objective
    "dropout1": trial.suggest_categorical("dropout1", [0.1, 0.2, 0.3, 0.4, 0.5]),
  File "/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/optuna/trial/_trial

KeyboardInterrupt: 

## Eval

In [ ]:
# test_drug = test_data.values[:, 0]
# test_cell = test_data.values[:, 1]

# test_labels = np.load("data/test_labels.npy")
# test_labels = torch.tensor(test_labels).float()
# test = [drug, cell, gene, edge_index, test_drug, test_cell, test_labels]

In [ ]:
# prob, res, test_attention = drGAT.eval(model, test)